# 08 — Head-to-head: in-kernel GEPA vs. subprocess GEPA

The honest question: is there a *measurable* ergonomic edge to running GEPA in a notebook vs. as a regular Python script? The LLM-bound optimizer takes about the same wall-clock either way — that's network-dominated. The gap shows up in the **surrounding work**: import time, dataset load, downstream analysis.

This notebook runs the *same* GEPA call twice — once via `python -c` (cold subprocess, what a script does) and once in-kernel (data + imports warm). Then it does a typical follow-up question and shows what the subprocess would have to redo.

> Reference: `~/Documents/GitHub/_docs/notebook/use-cases/01-gepa.md` § "Why nteract fits the GEPA loop."

In [1]:
import os, sys, time, subprocess, json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

import gepa
from gepa.examples.aime import init_dataset

TASK_LM = "bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0"
REFLECTION_LM = "bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0"
PY = sys.executable
print("python:", PY)

python: /Users/mhuang/.cache/uv/builds-v0/.tmpP6MEgE/bin/python


## 1. The benchmark script — what a CLI user would write

In [2]:
SCRIPT = Path("/tmp/abook_bench_subprocess.py")
SCRIPT.write_text(r'''
"""Stand-alone GEPA benchmark: identical params to the in-kernel cell."""
import os, time, json, sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv("/Users/mhuang/Documents/GitHub/abook/.env")

t_start = time.time()
import gepa
from gepa.examples.aime import init_dataset
t_after_import = time.time()

trainset_full, valset_full, _ = init_dataset()
t_after_load = time.time()

trainset = trainset_full[:3]
valset   = valset_full[:3]
seed = {"system_prompt": "Solve the math problem step by step. End with '### <number>'."}

result = gepa.optimize(
    seed_candidate=seed, trainset=trainset, valset=valset,
    task_lm="bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0",
    reflection_lm="bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    max_metric_calls=12, reflection_minibatch_size=3,
    skip_perfect_score=False, display_progress_bar=False, seed=0,
)
t_after_optimize = time.time()

# Subprocess MUST serialize the result if anything downstream wants it
out_path = Path("/tmp/abook_bench_result.json")
out_path.write_text(json.dumps({
    "num_candidates": result.num_candidates,
    "best_idx": result.best_idx,
    "val_aggregate_scores": result.val_aggregate_scores,
    "discovery_eval_counts": result.discovery_eval_counts,
    "best_prompt": result.best_candidate["system_prompt"],
}, indent=2))
t_after_dump = time.time()

# Print phase timings so the parent can read them
print(json.dumps({
    "phase_import": t_after_import - t_start,
    "phase_dataset": t_after_load - t_after_import,
    "phase_optimize": t_after_optimize - t_after_load,
    "phase_dump": t_after_dump - t_after_optimize,
    "phase_total": t_after_dump - t_start,
}))
''')
print("wrote", SCRIPT, f"({SCRIPT.stat().st_size} bytes)")

wrote /tmp/abook_bench_subprocess.py (1710 bytes)


## 2. Subprocess timing — `python script.py`

In [3]:
t0 = time.time()
proc = subprocess.run([PY, str(SCRIPT)], capture_output=True, text=True, timeout=600)
subprocess_total = time.time() - t0
print(f"subprocess wall-clock: {subprocess_total:.1f}s")
print(f"return code         : {proc.returncode}")
print()
print("--- last line of stdout (the phase JSON) ---")
phase_line = proc.stdout.strip().splitlines()[-1]
sub_phases = json.loads(phase_line)
for k, v in sub_phases.items():
    print(f"  {k:<20} {v:>6.2f}s")

subprocess wall-clock: 78.1s
return code         : 0

--- last line of stdout (the phase JSON) ---
  phase_import           0.03s
  phase_dataset          2.92s
  phase_optimize        74.75s
  phase_dump             0.00s
  phase_total           77.71s


## 3. In-kernel timing — gepa + AIME already loaded once

Pre-load the same dataset (timed). Then run optimize once (timed) — this is the equivalent of "I want to try another variant" in a notebook.

In [4]:
# Phase 1: dataset load (only happens once in a notebook — amortized)
t0 = time.time()
trainset_full, valset_full, _ = init_dataset()
t_load = time.time() - t0
print(f"dataset load (one-time): {t_load:.2f}s")

trainset = trainset_full[:3]
valset = valset_full[:3]
seed = {
    "system_prompt": "Solve the math problem step by step. End with '### <number>'."
}

# Phase 2: optimize (the LLM-bound part, same as subprocess)
t0 = time.time()
result = gepa.optimize(
    seed_candidate=seed,
    trainset=trainset,
    valset=valset,
    task_lm=TASK_LM,
    reflection_lm=REFLECTION_LM,
    max_metric_calls=12,
    reflection_minibatch_size=3,
    skip_perfect_score=False,
    display_progress_bar=False,
    seed=0,
)
in_kernel_optimize = time.time() - t0
print(f"optimize wall-clock    : {in_kernel_optimize:.2f}s")

ownloads.


dataset load (one-time): 3.26s
Iteration 0: Base program full valset score: 0.0 over 3 / 3 examples
Iteration 1: Selected program 0 score: 0.0
Iteration 1: Proposed new text for system_prompt: Solve the math problem step by step, showing all your work and reasoning clear
ly. After completing your solution, you MUST end your response with the final answer in the exact format: ### <number>

Important formatting requirements:
- The final line must be exactly: ### <number> where <number> is your numerical answer
- Do NOT use \boxed{} notation for the final answer
- Do NOT use any other formatting for the final answer line
- The ### must be on its own line at the very end

Solution guidelines:
- Break down the problem into clear, logical steps
- Show all calculations and intermediate results
- Explain your reasoning at each step
- For geometry problems, consider using coordinate systems when appropriate
- For counting problems, consider using Euler's formula (V - E + F = 2) for planar graph

ownloads.


## 4. The follow-up tax — five quick questions about `result`

For each downstream question we ask, the subprocess world has to re-run `python -c "..."` that imports gepa, reads the result JSON, computes the answer. In-kernel, the result is right there.

In [5]:
QUESTIONS = [
    ("num_candidates", "len(result.candidates)"),
    ("best_idx", "result.best_idx"),
    ("val_aggregate", "result.val_aggregate_scores"),
    ("best_prompt_len", "len(result.best_candidate['system_prompt'])"),
    ("parents", "result.parents"),
]

# In-kernel: each question is a dict lookup or attribute access
in_kernel_times = []
for label, expr in QUESTIONS:
    t0 = time.time()
    answer = eval(expr)
    in_kernel_times.append(time.time() - t0)
    print(
        f"  in-kernel {label:<18} -> {str(answer)[:50]}    ({in_kernel_times[-1] * 1000:.2f} ms)"
    )

print(f"\nin-kernel total for 5 questions: {sum(in_kernel_times) * 1000:.2f} ms")

  in-kernel num_candidates     -> 2    (0.02 ms)
  in-kernel best_idx           -> 0    (0.02 ms)
  in-kernel val_aggregate      -> [0.0, 0.0]    (0.01 ms)
  in-kernel best_prompt_len    -> 61    (0.03 ms)
  in-kernel parents            -> [[None], [0]]    (0.01 ms)

in-kernel total for 5 questions: 0.08 ms


In [6]:
import shlex

ANSWER_EXPRS = {
    "num_candidates": "len(r['discovery_eval_counts'])",
    "best_idx": "r['best_idx']",
    "val_aggregate": "r['val_aggregate_scores']",
    "best_prompt_len": "len(r['best_prompt'])",
    "parents": "[]  # not in subprocess JSON dump above; would need a re-run with more dumped fields",
}

subprocess_times = {}
for label, expr in ANSWER_EXPRS.items():
    cmd = (
        f'{PY} -c "'
        f"import json,sys; "
        f"r=json.loads(open('/tmp/abook_bench_result.json').read()); "
        f'print({expr})"'
    )
    t0 = time.time()
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    dt = time.time() - t0
    subprocess_times[label] = dt
    answer = out.stdout.strip()[:50]
    print(f"  subprocess {label:<18} -> {answer:<50} ({dt * 1000:.0f} ms)")

print(
    f"\nsubprocess total for 5 questions: {sum(subprocess_times.values()) * 1000:.0f} ms"
)

  subprocess num_candidates     -> 1                                                  (39 ms)
  subprocess best_idx           -> 0                                                  (32 ms)
  subprocess val_aggregate      -> [0.0]                                              (34 ms)
  subprocess best_prompt_len    -> 61                                                 (33 ms)
  subprocess parents            ->                                                    (28 ms)

subprocess total for 5 questions: 166 ms


## 5. The verdict

| Operation | Subprocess (cold) | In-kernel (warm) | Notes |
|-----------|---:|---:|---|
| Library import | ~30 ms | 0 ms (one-time) | gepa is already loaded |
| AIME dataset load | ~3 s | 0 ms (one-time) | HuggingFace caches but disk reload still costs |
| `gepa.optimize` | ~75 s | ~80 s | LLM-bound, dominated by Bedrock latency — basically a tie |
| Result write (JSON) | ~0 ms | 0 ms | (in-kernel doesn't need it) |
| **5 follow-up questions** | **166 ms** | **0.08 ms** | **~2000× faster, and not constrained to what got dumped** |

The optimizer itself is the same on both substrates. The notebook win is **everything around it**: imports stay warm, the dataset stays loaded, the full `GEPAResult` (not just whatever you remembered to dump to JSON) is always one attribute away.

The subprocess version of `num_candidates` was `1` (its own independent run) while the in-kernel one was `2` — the runs are different samples. That's also a notebook win: you can compare the two right here, in the same scope, without writing a "compare two result JSONs" script.

## 6. What you can do now that you couldn't with subprocesses

Just one example — slice the val_subscores DataFrame from the in-kernel result by candidate index:

In [7]:
import pandas as pd

df = pd.DataFrame(result.val_subscores).T
df.columns = [f"cand {i}" for i in range(result.num_candidates)]
df.index.name = "val_instance"
print(
    "In-kernel: val_subscores as a DataFrame (the subprocess JSON didn't include this)"
)
df

In-kernel: val_subscores as a DataFrame (the subprocess JSON didn't include this)


,cand 0,cand 1
val_instance,,
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0


## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. The benchmark script — what a CLI user would write
- 2. Subprocess timing — `python script.py`
- 3. In-kernel timing — gepa + AIME already loaded once
- 4. The follow-up tax — five quick questions about `result`
- 5. The verdict
- 6. What you can do now that you couldn't with subprocesses

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import json as _json
from pathlib import Path as _Path
import collections as _c

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/gepa/08-vs-script-baseline.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print(f"output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")


## Data sources

| Source | Path |
|---|---|
| Benchmark script | `/tmp/abook_bench_subprocess.py` (written by this notebook) |
| Subprocess result | `/tmp/abook_bench_result.json` |
| In-kernel result  | `result` object in this notebook's namespace |
| Eval data | HuggingFace `AI-MO/aimo-validation-aime` (3 train + 3 val real items) |
| Task LM | `bedrock/converse/us.anthropic.claude-haiku-4-5-20251001-v1:0` |
| Reflection LM | `bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0` |
| Timings | Real `time.time()` measurements from this session |

→ **Done.** This is the last notebook in the series. See `README.md` in this folder for the full map.